In [1]:
import sys
sys.path.append("..")

import numpy as np
import wfdb
from sklearn.metrics import classification_report
from src.pipeline import load_model, analyze, CLASSES, AAMI_MAP

bundle = load_model("../models/ecg_rf.joblib")
DATA_DIR = "../data/mitdb"

def evaluate(rec):
    record = wfdb.rdrecord(f"{DATA_DIR}/{rec}")
    ann = wfdb.rdann(f"{DATA_DIR}/{rec}", "atr")
    ch = record.sig_name.index("MLII")
    res = analyze(record.p_signal[:, ch], record.fs, bundle)

    keep = [i for i, s in enumerate(ann.symbol)
            if s in AAMI_MAP and AAMI_MAP[s] in CLASSES]
    a_samples = ann.sample[keep]
    a_labels = np.array([CLASSES.index(AAMI_MAP[ann.symbol[i]]) for i in keep])

    y_true, y_pred = [], []
    for p, c in zip(res["peaks"], res["pred"]):
        j = np.argmin(np.abs(a_samples - p))
        if abs(a_samples[j] - p) <= 36:      # حداکثر ۰٫۱ ثانیه فاصله
            y_true.append(a_labels[j])
            y_pred.append(c)

    print(f"===== record {rec}")
    print("ضربان‌های برچسب‌دار:", len(a_samples),
          "| پیدا شده:", len(res["peaks"]),
          "| جور شده:", len(y_true))
    print(classification_report(y_true, y_pred, labels=[0, 1, 2, 3],
                                target_names=CLASSES, digits=3, zero_division=0))

for rec in ["100", "200", "233"]:
    evaluate(rec)

===== record 100
ضربان‌های برچسب‌دار: 2273 | پیدا شده: 2271 | جور شده: 2271
              precision    recall  f1-score   support

           N      1.000     1.000     1.000      2237
           S      1.000     1.000     1.000        33
           V      1.000     1.000     1.000         1
           F      0.000     0.000     0.000         0

    accuracy                          1.000      2271
   macro avg      0.750     0.750     0.750      2271
weighted avg      1.000     1.000     1.000      2271

===== record 200
ضربان‌های برچسب‌دار: 2601 | پیدا شده: 2598 | جور شده: 2595
              precision    recall  f1-score   support

           N      0.987     0.983     0.985      1742
           S      0.266     0.833     0.403        30
           V      1.000     0.929     0.963       822
           F      0.000     0.000     0.000         1

    accuracy                          0.964      2595
   macro avg      0.563     0.686     0.588      2595
weighted avg      0.982     0.964